# Function Calling
Function Calling은 기본적으로 챗봇이 사용자에게 **어떤 기능(function)을 실행하기 위한 파라미터**를 계속 요청하는 기법을 뜻합니다.

**예시**
- 사용자에게 id를 입력 받아 주문 배송 일자를 리턴해주는 챗봇
- 사용자에게 책상의 가로,세로,깊이를 입력 받아 상품을 추천해주는 챗봇

사용자로부터 기능을 실행하기 위한 입력값을 받기 위해 챗봇은 끊임없이 질문을 던져야 합니다.

이 때 주의해야 할 점은 챗봇이 직접 개발자가 원하는 함수를 호출하는 것이 아니라는 것입니다. **⭐️단순히 어떤 함수가 호출 될 준비가 되었다⭐️**라는 것만 알려줍니다. 즉 함수 호출은 개발자가 **직접**해야 한다는 것입니다.

In [1]:
!pip install openai langchain langchain_community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.2/325.2 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.4/404.4 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.8/295.8 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141.9 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/

# OpenAI 설정

In [2]:
import os
import json
from datetime import datetime
import openai

In [ ]:
OPENAI_API_KEY = "~~~~~~~~~"
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

In [4]:
client = openai.OpenAI()

In [7]:
# 챗봇이 이 함수를 호출해라! 라고 알려줍니다.
def get_delivery_date(order_id: str ) -> datetime:
    # 어쩌고 저쩌고 하면서 데이터베이스에 접근 하는 코드가 여기에 있으면 된다.
    #  리턴 되는건 db 조회 결과를 리턴 시키면 됩니다.
    return datetime.today().strftime('%Y-%m-%d')

In [8]:
# LLM에서 사용할 함수 tool 생성

delivery_tool = {
    "type" : "function", # 도구의 타입을 function으로 설정하여 함수 호출 기능 제공을 명시

    "function": {
        "name" : "get_delivery_data", # 함수의 이름을 정의. 이 이름은 챗봇이 개발자에게 호출하라고 알려주는 역할

        # 함수의 목적과 사용 사례를 설명
            # 챗봇이 언제 이 함수를 호출해야 하는지 명확히 알 수 있도록 설명을 포함
        "description" : "고객의 주문에 대한 배송 날짜를 확인합니다. 예를 들어, 고객이 '내 패키지가 어디에 있나요?'라고 물을 때 이 함수를 호출하세요.",

        # 함수가 호출되어야 할 파라미터에 대한 설명
        "parameters" : {
            "type" : "object", # 일반적으로 object 타입으로 함수가 받아야 할 파라미터 타입을 설정
            "properties" : {
                "order_id" : {
                    "type" : "string",
                    "description" : "고객의 주문 ID"
                                }
                            },
            "required" : ["order_id"], # 함수 호출 시 반드시 제공되어야 하는 필수 파라미터
            "additionalProperties" : False # 추가적인 파라미터는 허용하지 않음을 명시
                        }
                }
            }

tools = [delivery_tool]

In [11]:
messages = []

messages.append({"role": "system", "content": "당신은 도움이 되는 고객 지원 어시스턴트입니다. 제공된 도구를 사용하여 사용자를 지원하세요."})    # 역할 부여 + 도구(함수)가 있음을 알려줌 => 도구로 사용자를 지원해라

messages.append({"role": "user", "content": "안녕하세요, 제 주문의 배송 날짜를 알려주실 수 있나요?"})

messages.append({"role": "assistant", "content": "안녕하세요! 제가 도와드릴 수 있습니다. 주문 ID를 알려주시겠어요?"})

messages.append({"role": "user", "content": "쓸데 없는 소리"})

In [13]:
response = client.chat.completions.create(
    model='gpt-4o',
    messages=messages,
    tools=tools # 지원 도구 설정
)

response.choices[0].message

ChatCompletionMessage(content='주문 ID를 제공해 주셔야 배송 날짜를 확인할 수 있습니다. 주문 ID를 알려주시면 바로 도와드리겠습니다.', refusal=None, role='assistant', function_call=None, tool_calls=None)

content='주문 ID를 제공해 주셔야 배송 날짜를 확인할 수 있습니다. 주문 ID를 알려주시면 바로 도와드리겠습니다.', refusal=None, role='assistant', function_call=None, tool_calls=None


In [16]:
messages = []
messages.append({"role": "system", "content": "당신은 도움이 되는 고객 지원 어시스턴트입니다. 제공된 도구를 사용하여 사용자를 지원하세요."})
messages.append({"role": "user", "content": "안녕하세요, 제 주문의 배송 날짜를 알려주실 수 있나요?"})
messages.append({"role": "assistant", "content": "안녕하세요! 제가 도와드릴 수 있습니다. 주문 ID를 알려주시겠어요?"})
messages.append({"role": "user", "content": "제 생각엔 order12345인 것 같아요"})

response = client.chat.completions.create(
    model='gpt-4o',
    messages=messages,
    tools=tools # 지원 도구 설정
)

response.choices[0].message

ChatCompletionMessage(content=None, refusal=None, role='assistant', function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_LwYZ11TpVlKDBDShOfhSIja5', function=Function(arguments='{"order_id":"order12345"}', name='get_delivery_data'), type='function')])

content=None, refusal=None, role='assistant', function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_LwYZ11TpVlKDBDShOfhSIja5', function=Function(arguments='{"order_id":"order12345"}', name='get_delivery_data'), type='function')]

In [21]:
# get_delivery_date 함수를 위한 매개변수를 추출 - 이게 없으면 Function Calli을 할 수 없는 상황

tool_call = response.choices[0].message.tool_calls

if tool_call:
    print(tool_call[0])
else:
    print("함수를 호출하기 위한 작업이 마무리되지 않았습니다.") # 사실 이게 있을 필요는 없다 -> 어차피 LLM이 계속 물어볼 거니까

ChatCompletionMessageToolCall(id='call_LwYZ11TpVlKDBDShOfhSIja5', function=Function(arguments='{"order_id":"order12345"}', name='get_delivery_data'), type='function')


In [22]:
arguments = json.loads(tool_call[0].function.arguments)
arguments

{'order_id': 'order12345'}

In [23]:
get_delivery_date(**arguments)

'2024-10-15'

In [24]:
if response.choices[0].finish_reason == "tool_calls":
  arguments = json.loads(response.choices[0].message.tool_calls[0].function.arguments)
  delivery_date = get_delivery_date(**arguments)

  print(delivery_date) # 이 내용을 Web UI 같은걸로 보여줄 수 있도록 코딩합니다.

2024-10-15


# Langchain

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.schema import SystemMessage, HumanMessage, AIMessage, ChatMessage

In [ ]:
tools = [{
    "name": "get_delivery_date",
    "description": "고객의 주문에 대한 배송 날짜를 확인합니다. 예를 들어, 고객이 '내 패키지가 어디에 있나요?'라고 물을 때 이 함수를 호출하세요.",
    "parameters": {
        "type": "object",
        "properties": {
            "order_id": {
                "type": "string",
                "description": "고객의 주문 ID."
            }
        },
        "required": ["order_id"]
    }
}]

In [ ]:
llm = ChatOpenAI(model="gpt-3.5-turbo")
message = llm.predict_messages(
    [SystemMessage(content="당신은 도움이 되는 고객 지원 어시스턴트입니다. 제공된 도구를 사용하여 사용자를 지원하세요."),
    HumanMessage(content="안녕하세요, 제 주문의 배송 날짜를 알려주실 수 있나요?"),
    AIMessage(content="안녕하세요! 제가 도와드릴 수 있습니다. 주문 ID를 알려주시겠어요"),
    HumanMessage(content="제 생각엔 order_12345인 것 같아요")], functions=tools
)

In [ ]:
message

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"order_id":"order_12345"}', 'name': 'get_delivery_date'}}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 237, 'total_tokens': 256}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'function_call', 'logprobs': None}, id='run-13baa1b2-5628-458d-aead-a85b556491ba-0')

In [ ]:
message.additional_kwargs

{'function_call': {'arguments': '{"order_id":"order_12345"}',
  'name': 'get_delivery_date'}}

In [ ]:
arguments = json.loads(message.additional_kwargs["function_call"]["arguments"])
arguments

{'order_id': 'order_12345'}

In [ ]:
# order_id를 전달하여 배달 일자를 전달받는다.
delivery_date = get_delivery_date(**arguments)
print(delivery_date)

2024-09-02


마찬가지로 `finish_reason`에 `function_call`이 있습니다.

In [ ]:
if message.response_metadata["finish_reason"] == "function_call":
  arguments = json.loads(message.additional_kwargs["function_call"]["arguments"])
  delivery_date = get_delivery_date(**arguments)

  print(delivery_date)

2024-09-02
